# Lab 01: Number Theory Foundations for Cryptography
**Student Name:** Md. Akib Al Jawad  
**Student ID:** IT-23020  
**Department:** Information and Communication Technology (ICT)  
**Institution:** Mawlana Bhashani Science and Technology University (MBSTU)  
**Reference:** IIT Bombay Maths and Physics Club — Number Theory & Cryptography Workshop (Day 1 Assignment)

---

## Abstract & Objectives
In modern public-key cryptography (e.g., RSA, Diffie-Hellman, Elliptic Curve Cryptography), security relies directly on the mathematical properties of integers in modular arithmetic. This notebook explores and implements the fundamental algorithms that power asymmetric cryptosystems:
1. **Extended Euclidean Algorithm**: Finding $\gcd(a, b)$ and Bézout coefficients $(x, y)$ such that $ax + by = \gcd(a, b)$.
2. **Least Common Multiple (LCM)**: Efficient computation via $\text{lcm}(a, b) = \frac{a \cdot b}{\gcd(a, b)}$.
3. **Linear Diophantine Equations**: Solving equations of the form $ax + by = c$.
4. **Modular Multiplicative Inverses & Modular Division**: Computing $a^{-1} \pmod n$ and solving $ax \equiv b \pmod n$.
5. **Fast Modular Exponentiation**: Binary square-and-multiply algorithm computing $b^e \pmod m$ in $O(\log e)$.
6. **Chinese Remainder Theorem (CRT)**: Uniquely reconstructing solutions to systems of simultaneous congruences with coprime moduli.



### Problem 1: Extended Euclidean Algorithm
The standard Euclidean algorithm relies on the recurrence $\gcd(a, b) = \gcd(b, a \bmod b)$.  
Bézout's identity guarantees that for any non-zero integers $a$ and $b$, there exist integers $x$ and $y$ such that:
$$a \cdot x + b \cdot y = \gcd(a, b)$$

In the inductive step, if $\gcd(b, a \bmod b) = b \cdot x_1 + (a \bmod b) \cdot y_1$, then substituting $a \bmod b = a - \lfloor a / b \rfloor \cdot b$ gives:
$$\gcd(a, b) = a \cdot y_1 + b \cdot (x_1 - \lfloor a / b \rfloor \cdot y_1)$$
Thus: $x = y_1$ and $y = x_1 - \lfloor a / b \rfloor \cdot y_1$.



In [1]:
def extended_gcd(a, b):
    assert a >= b and b >= 0 and a + b > 0, "Requires a >= b >= 0 and a + b > 0"
    if b == 0:
        d, x, y = a, 1, 0
    else:
        d, x1, y1 = extended_gcd(b, a % b)
        x = y1
        y = x1 - (a // b) * y1
    
    assert a % d == 0 and b % d == 0
    assert d == a * x + b * y
    return (d, x, y)

# Demonstration and validation
res1 = extended_gcd(8, 4)
res2 = extended_gcd(27, 15)
print(f"extended_gcd(8, 4)   = {res1} -> 8*({res1[1]}) + 4*({res1[2]}) = {res1[0]}")
print(f"extended_gcd(27, 15) = {res2} -> 27*({res2[1]}) + 15*({res2[2]}) = {res2[0]}")



extended_gcd(8, 4)   = (4, 0, 1) -> 8*(0) + 4*(1) = 4


extended_gcd(27, 15) = (3, 2, -3) -> 27*(2) + 15*(-3) = 3


### Problem 2: Least Common Multiple (LCM)
The connection between GCD and LCM for two positive integers $a, b$ is given by:
$$a \cdot b = \gcd(a, b) \cdot \text{lcm}(a, b) \implies \text{lcm}(a, b) = \frac{a \cdot b}{\gcd(a, b)}$$



In [1]:
def gcd_simple(a, b):
    while b:
        a, b = b, a % b
    return abs(a)

def lcm(a, b):
    assert a > 0 and b > 0
    return (a * b) // gcd_simple(a, b)

assert lcm(9, 3) == 9
assert lcm(17, 37) == 629
print(f"lcm(9, 3)   = {lcm(9, 3)}")
print(f"lcm(17, 37) = {lcm(17, 37)}")



lcm(9, 3)   = 9


lcm(17, 37) = 629


### Problem 3: Solving Linear Diophantine Equations
Given $ax + by = c$, a solution in integers $(x, y)$ exists if and only if $d = \gcd(a, b)$ divides $c$.  
Using Bézout's identity, we obtain $x_0, y_0$ such that $ax_0 + by_0 = d$.  
Multiplying throughout by $k = c / d$:
$$a(x_0 \cdot k) + b(y_0 \cdot k) = c$$



In [1]:
def diophantine(a, b, c):
    assert a > 0 and b > 0
    if a >= b:
        gcd_val, x0, y0 = extended_gcd(a, b)
    else:
        gcd_val, y0, x0 = extended_gcd(b, a)
        
    assert c % gcd_val == 0, f"No integer solution because gcd({a}, {b}) does not divide {c}"
    scale = c // gcd_val
    x = x0 * scale
    y = y0 * scale
    
    assert a * x + b * y == c
    return (x, y)

sol1 = diophantine(34, 51, 170)
sol2 = diophantine(91, 117, 65)
print(f"diophantine(34, 51, 170) -> (x, y) = {sol1} | Check: 34*{sol1[0]} + 51*{sol1[1]} = {34*sol1[0] + 51*sol1[1]}")
print(f"diophantine(91, 117, 65) -> (x, y) = {sol2} | Check: 91*{sol2[0]} + 117*{sol2[1]} = {91*sol2[0] + 117*sol2[1]}")



diophantine(34, 51, 170) -> (x, y) = (-10, 10) | Check: 34*-10 + 51*10 = 170


diophantine(91, 117, 65) -> (x, y) = (20, -15) | Check: 91*20 + 117*-15 = 65


### Problem 4: Modular Multiplicative Inverse & Modular Division
If $\gcd(a, n) = 1$, then by Bézout's identity:
$$a \cdot x + n \cdot y = 1 \implies a \cdot x \equiv 1 \pmod n$$
Here $x$ is the unique multiplicative inverse $a^{-1} \pmod n$.  
To compute $b / a \pmod n$, we multiply by the inverse: $x = (b \cdot a^{-1}) \pmod n$.



In [1]:
def modInverse(a, n):
    """Calculate the multiplicative inverse of a (mod n)"""
    a = a % n
    d, yn, ya = extended_gcd(n, a)
    assert d == 1, "Inverse does not exist: gcd(a, n) != 1"
    return ya % n

def divide(b, a, n):
    assert n > 1 and a > 0
    aInv = modInverse(a, n)
    assert (a * aInv) % n == 1
    return (aInv * b) % n

print(f"divide(9, 5, 3)    = {divide(9, 5, 3)}")
print(f"divide(223, 67, 54) = {divide(223, 67, 54)}")



divide(9, 5, 3)    = 0


divide(223, 67, 54) = 13


### Problem 5: Fast Modular Exponentiation (Square-and-Multiply)
Naively calculating $b^e \bmod m$ requires $O(e)$ multiplications, which is impossible for 2048-bit cryptographic exponents.  
The **binary square-and-multiply** algorithm processes the bits of $e$ from least to most significant (or vice-versa), squaring the base each step and multiplying whenever the current bit is 1. This computes $b^e \pmod m$ in at most $2 \lfloor \log_2 e \rfloor$ modular multiplications.



In [1]:
def FastModularExponentiation(b, e, m):
    if m == 1:
        return 0
    res = 1
    base = b % m
    exp = e
    while exp > 0:
        if exp % 2 == 1:
            res = (res * base) % m
        base = (base * base) % m
        exp //= 2
    return res

val = FastModularExponentiation(4, 5, 3)
assert val == pow(4, 5, 3)
print(f"FastModularExponentiation(4, 5, 3) = {val}")

# Large integer test
large_b = 3141592653589793
large_e = 12345678910111213
large_m = 1000000007
large_res = FastModularExponentiation(large_b, large_e, large_m)
assert large_res == pow(large_b, large_e, large_m)
print(f"FastModularExponentiation({large_b}, {large_e}, {large_m}) = {large_res}")



FastModularExponentiation(4, 5, 3) = 1


FastModularExponentiation(3141592653589793, 12345678910111213, 1000000007) = 765976732


### Problem 6: Chinese Remainder Theorem (CRT)
Given coprime moduli $n_1, n_2$ with $\gcd(n_1, n_2) = 1$, the system:
$$\begin{cases} r \equiv r_1 \pmod{n_1} \\ r \equiv r_2 \pmod{n_2} \end{cases}$$
has a unique solution modulo $n_1 n_2$.  
From Bézout's identity, $n_1 x + n_2 y = 1$, which implies:
$$n_2 y \equiv 1 \pmod{n_1} \quad \text{and} \quad n_1 x \equiv 1 \pmod{n_2}$$
Thus, the solution is constructed as:
$$r = (r_1 \cdot n_2 y + r_2 \cdot n_1 x) \bmod (n_1 n_2)$$



In [1]:
def ChineseRemainderTheorem(n1, r1, n2, r2):
    (d, x, y) = extended_gcd(n1, n2)
    assert d == 1
    modulus = n1 * n2
    r = (r1 * n2 * y + r2 * n1 * x) % modulus
    return (r + modulus) % modulus

crt1 = ChineseRemainderTheorem(13, 7, 6, 5)
crt2 = ChineseRemainderTheorem(28, 20, 19, 3)

assert crt1 == 59
assert crt2 == 440
print(f"ChineseRemainderTheorem(13, 7, 6, 5)   = {crt1} -> (59%13 = {crt1%13}, 59%6 = {crt1%6})")
print(f"ChineseRemainderTheorem(28, 20, 19, 3) = {crt2} -> (440%28 = {crt2%28}, 440%19 = {crt2%19})")



ChineseRemainderTheorem(13, 7, 6, 5)   = 59 -> (59%13 = 7, 59%6 = 5)


ChineseRemainderTheorem(28, 20, 19, 3) = 440 -> (440%28 = 20, 440%19 = 3)
